# 03 — Build sliding windows

**Goal:** turn a sequence of IDs into next-token prediction examples.

We start with hand-picked integers, then use the tokenizer and splits prepared in earlier notebooks. We manipulate Python lists only: no model is trained and PyTorch is not needed.

By the end, you will understand:

- why an input of length **L** needs **L + 1** available IDs;
- how the target offset and `stride` work;
- how to group examples into a batch;
- how to preserve document boundaries and the train/validation split.

The numeric examples are invented for teaching. Real files are loaded in the second part; this notebook does not download anything.

## 1. A sequence of IDs

The numbers below represent tokens. Their numeric values do not say anything about their meaning.

At each position, the model will predict the next number. Here we only prepare inputs and targets.

In [ ]:
ids = [10, 20, 30, 40, 50, 60]
length = 3

x = ids[:length]
y = ids[1:length + 1]

print("Input:", x)  # [10, 20, 30]
print("Target:", y)  # [20, 30, 40]
assert len(x) == len(y) == length
assert x[1:] == y[:-1]

### Why L + 1?

For the input `[10, 20, 30]`, the three expected targets are `[20, 30, 40]`. The fourth ID is needed for the last target.

The **stride** tells us how many positions to advance before the next window. It does not change the target offset, which remains **one token**.

In `range(0, len(ids) - length, stride)`, the right endpoint is exclusive. This ensures that the final target exists.

In [ ]:
def windows(ids, length=8, stride=4):
    """Produce complete (input, target) pairs from a sequence."""
    if length < 1 or stride < 1:
        raise ValueError("length and stride must be positive.")

    for start in range(0, len(ids) - length, stride):
        x = ids[start:start + length]
        y = ids[start + 1:start + length + 1]
        yield x, y

In [ ]:
examples = list(windows(ids, length=3, stride=2))
assert examples == [
    ([10, 20, 30], [20, 30, 40]),
    ([30, 40, 50], [40, 50, 60]),
]

for number, (x, y) in enumerate(examples, start=1):
    print(f"Window {number} : X = {x} ; Y = {y}")

# With L IDs, the final target is missing.
assert list(windows([10, 20, 30], length=3, stride=1)) == []

# With L + 1 IDs, one window is possible.
assert list(windows([10, 20, 30, 40], length=3, stride=1)) == [
    ([10, 20, 30], [20, 30, 40])
]

## 2. Observe the stride

With `stride=1`, windows overlap heavily. With `stride=length`, consecutive input blocks do not overlap; their targets are still shifted by one token.

This version keeps only complete windows. A short remainder at the end is not padded artificially. A larger stride can leave some positions unused.

In [ ]:
sequence = list(range(9))

for stride in (1, 2, 3):
    result = list(windows(sequence, length=3, stride=stride))
    print(f"stride={stride}: {len(result)} windows")
    for x, y in result:
        print("  X =", x, "Y =", y)

## 3. Group examples into batches

A **batch** is a group of examples. For a batch size B and window length L, `X` and `Y` are lists of B lists, each containing L integers.

The final batch may be smaller than B. Set `drop_last=True` to discard it. At this stage, we do not compute predictions or update parameters.

In [ ]:
def batches(examples, batch_size=4, drop_last=False):
    """Group (x, y) pairs without changing the windows."""
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")

    batch_x, batch_y = [], []
    for x, y in examples:
        batch_x.append(x)
        batch_y.append(y)
        if len(batch_x) == batch_size:
            yield batch_x, batch_y
            batch_x, batch_y = [], []

    if batch_x and not drop_last:
        yield batch_x, batch_y

In [ ]:
numeric_examples = list(windows(list(range(8)), length=3, stride=1))
numeric_batches = list(batches(numeric_examples, batch_size=2))

assert [len(batch_x) for batch_x, _ in numeric_batches] == [2, 2, 1]

first_x, first_y = numeric_batches[0]
print("X :", first_x)
print("Y :", first_y)

assert first_x == [[0, 1, 2], [1, 2, 3]]
assert first_y == [[1, 2, 3], [2, 3, 4]]

## 4. Locate the project artifacts

We use the files prepared in the earlier notebooks:

- `artifacts/tokenizer/tokenizer_darija.json`;
- `data/processed/train.jsonl`;
- `data/processed/validation.jsonl`.

Each JSONL line must be an object with a string field named `text`. The splits are already defined: **we do not mix documents or split them again here**.

The kernel needs the `tokenizers` library used in notebook 02. This notebook does not download data or train the tokenizer.

In [ ]:
import json
import unicodedata
from pathlib import Path


def find_project_root(start=None):
    """Search for README.md and notebooks/ from cwd, then in its parent directories."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Project root not found: open this notebook from the repository "
        "containing README.md and notebooks/."
    )


PROJECT_ROOT = find_project_root()
TOKENIZER_PATH = PROJECT_ROOT / "artifacts/tokenizer/tokenizer_darija.json"
TRAIN_PATH = PROJECT_ROOT / "data/processed/train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data/processed/validation.jsonl"

missing = [
    str(path.relative_to(PROJECT_ROOT))
    for path in (TOKENIZER_PATH, TRAIN_PATH, VALIDATION_PATH)
    if not path.is_file()
]
if missing:
    raise FileNotFoundError(
        "Fichiers manquants : " + ", ".join(missing)
        + ". Prepare them with the previous notebooks before continuing."
    )

print("Project root:", PROJECT_ROOT)

### Load the existing tokenizer

We restore `encode_special_tokens=True` after loading so literal strings such as `[EOS]` remain ordinary text. In Tokenizers 0.22.1, this flag is not stored in the JSON file.

The real EOS marker is added **by its ID**, after encoding each document. Reserving a special token in the vocabulary does not add it automatically.

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.encode_special_tokens = True

eos_id = tokenizer.token_to_id("[EOS]")
if eos_id is None:
    raise ValueError("The tokenizer must reserve the [EOS] special token.")

print("Vocabulary size:", tokenizer.get_vocab_size())
print("EOS ID:", eos_id)

## 5. Read the splits without mixing them

We read the local files and preserve document boundaries. NFC is applied in the same way to both splits; no vocabulary is learned here.

The intersection below detects documents that are exactly identical after NFC. It does not detect paraphrases or near-duplicates; grouping those belongs in corpus preparation.

In [ ]:
def read_documents(path):
    """Read text from a local JSONL file, one document per line."""
    with Path(path).open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict) or not isinstance(
                record.get("text"), str
            ):
                raise ValueError(
                    f"{Path(path).name}, ligne {line_number} : "
                    "an object with a string text field is expected."
                )
            text = unicodedata.normalize("NFC", record["text"])
            if not text.strip():
                raise ValueError(
                    f"{Path(path).name}, ligne {line_number} : texte vide."
                )
            yield text


train_documents = list(read_documents(TRAIN_PATH))
validation_documents = list(read_documents(VALIDATION_PATH))

if not train_documents or not validation_documents:
    raise ValueError("Both splits must contain documents.")

overlap = set(train_documents).intersection(validation_documents)
if overlap:
    raise ValueError(
        f"{len(overlap)} document(s) identique(s) dans les deux partitions. "
        "Fix the corpus preparation before continuing."
    )

print("Training documents:", len(train_documents))
print("Validation documents:", len(validation_documents))

## 6. Build windows one document at a time

For each document:

1. encode the text with the existing rules;
2. add exactly one real EOS ID at the end;
3. produce its complete windows;
4. move to the next document.

We never concatenate IDs from two documents. A target from one document therefore cannot be the start of the next.

A three-token text can produce a length-3 window **after EOS is added**, because the sequence then has four IDs.

In [ ]:
def document_windows(tokenizer, documents, length=8, stride=4):
    """Encode and split each document separately, with a final EOS."""
    eos = tokenizer.token_to_id("[EOS]")
    if eos is None:
        raise ValueError("The tokenizer does not contain [EOS].")

    for text in documents:
        ids = tokenizer.encode(text, add_special_tokens=False).ids
        ids = ids + [eos]
        yield from windows(ids, length=length, stride=stride)

## 7. Prepare one batch from each split

We use the same length and stride for both splits. Training and validation windows remain in separate generators.

We take only the first batch from each generator to inspect its structure. When you run these cells, results will come from your files; no real values are prefilled.

In [ ]:
LENGTH = 16
STRIDE = 8
BATCH_SIZE = 4

train_examples = document_windows(
    tokenizer, train_documents, length=LENGTH, stride=STRIDE
)
validation_examples = document_windows(
    tokenizer, validation_documents, length=LENGTH, stride=STRIDE
)

first_train_batch = next(batches(train_examples, BATCH_SIZE), None)
first_validation_batch = next(batches(validation_examples, BATCH_SIZE), None)

for name, batch in (
    ("Training", first_train_batch),
    ("Validation", first_validation_batch),
):
    if batch is None:
        print(name, ": no complete window at this length.")
        continue

    batch_x, batch_y = batch
    assert len(batch_x) == len(batch_y)
    for x, y in zip(batch_x, batch_y):
        assert len(x) == len(y) == LENGTH
        assert x[1:] == y[:-1]
        assert eos_id not in x
        assert eos_id not in y[:-1]

    print(name, ": list shapes =", (len(batch_x), LENGTH))
    print("First input :", batch_x[0])
    print("First target  :", batch_y[0])

## Key points

- The target is always shifted by one token; `stride` controls the window starts.
- The splits stay separate, and no window crosses a document boundary.
- Texts that are too short do not produce a complete window.
- The stride may leave the end of a document unused. Adding EOS does not guarantee that it appears in retained targets. With `stride=1`, the last complete window reaches EOS when the document is long enough.
- A truncated window can split the bytes of a character. Here we check ID lists and do not require each window decoded in isolation to reconstruct complete text.
- These batches are prepared inputs for a future model. They are not a prediction or a training run.

When `LENGTH` increases, some documents stop producing windows because each example needs `LENGTH + 1` IDs, including EOS.

Next, we will map IDs to vectors with embeddings.